In [1]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
from torch import float32, tensor
from torch.nn import CrossEntropyLoss, Linear, Sequential, Softmax
from torch.nn.functional import softmax, cross_entropy
from torch.optim import SGD


import import_ipynb
from common import assert_eq, Patient, T # type: ignore


def logistic_regression_nc_sgd_autograd(X, y, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with autograd.

    Args:
        X: Input features of shape (Samples, Features)
        y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """
    ((s, f), c) = (X.shape, len(set(y.squeeze().tolist())))

    assert_eq(X.shape, (s, f))

    model = Linear(f, c)
    loss_fn = cross_entropy

    optimizer = SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        logits = model(X)
        assert_eq(logits.shape, (s, c))

        loss = loss_fn(logits, y.squeeze().long())
        assert_eq(loss.shape, ())
        
        loss.backward()
        optimizer.step()

    return (loss.item(), model)


def _test_logistic_regression_nc_sgd_autograd(epochs=2000, lr=0.1) -> None:
    training_data = T([Patient([0.5, 0.25, 0.25]).data for _ in range(70)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling    
    y = training_data[:, -1].unsqueeze(1)

    (_, model) = logistic_regression_nc_sgd_autograd(X, y, epochs, lr)

    predict = lambda x: softmax(model(x), dim=0).argmax()

    for d in ([Patient([1.0, 0.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 1.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 0.0, 1.0]).data for _ in range(10)]):
        x = T(d[:-1])
        y = T(d[-1])

        x[0] /= 100
        x[5] /= 200
        assert_eq(predict(x), y)

    
if __name__ == "__main__":
    _test_logistic_regression_nc_sgd_autograd()

TypeError: linear(): argument 'input' (position 1) must be Tensor, not list